# Compare Jae-Mo's `write_ndegen_applied` `_r.dat`

`compare_pr702_rdat.ipynb` assumes the candidate and reference share an identical `(R, n, m)` row layout, which holds for the canonical `use_ws_distance=F` grid. It does not hold here: Jae-Mo's `plan5-write-ndegen-applied` branch (`jaemolihm/wannier90`), when run with `transl_inv_full=T, use_ws_distance=T`, requires `write_ndegen_applied=T` and then emits a *folded* R list -- a Wigner-Seitz-degenerate bond can appear at more than one R image, each row already carrying its `1/ndegen` weight (self-contained, unlike the classic `_hr.dat` + `_wsvec.dat` convention). postw90's own `write_aa_r` reference reorders (and, empirically, also lengthens) its row list at `use_ws_distance=T` for the same reason, so both files must be matched by key rather than by position.

Unlike `compare_pr702_rdat.ipynb`, there is no fixed production candidate/reference pair here -- both files come from ad hoc scratch runs (see `notes/progress/2026-09-13_jaemo_transl_inv_full_ws_distance.md`), so `CASES` below starts empty; fill in real paths before setting `RUN_COMPUTATION = True`.

No machine-readable artifact was ever saved for this check -- only plain stdout captures (`results/jaemo_rdat_file/wsT_keyed_basic2.txt`, `wsT_keyed_diamond.txt`) from past ad hoc runs on the `diamond`/`testw90_example05` and `testw90_basic2` systems (see `notes/progress/2026-09-13_jaemo_transl_inv_full_ws_distance.md` for the full write-up: both bit-exact after 6-decimal rounding, 100.000000%, 0 unmatched keys). `RUN_COMPUTATION = False` below just prints those historical captures rather than reconstructing structured results.

In [ ]:
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS
from validation.inputs.compare_pr702_rdat_lib import read_matrix, fortran_round

## Config

`CASES` is `(label, candidate_path, reference_path)`. Add real paths from a `write_ndegen_applied`/`write_aa_r` scratch run before switching `RUN_COMPUTATION` on.

In [ ]:
RUN_COMPUTATION = False  # True: compare real files below. False: print the historical stdout captures in results/jaemo_rdat_file/.
OUTPUT_DIR = RESULTS / "jaemo_rdat_file"
PRINT_STEP = 1e-6
CASES = [
    # ("diamond", Path("/path/to/diamond_r.dat"), Path("/path/to/diamond_r_full.dat")),
    # ("basic2", Path("/path/to/testw90_basic2_r.dat"), Path("/path/to/testw90_basic2_r_full.dat")),
]

## Comparison functions

Keyed by `(R, n, m)` rather than by row position, since the two files fold Wigner-Seitz-degenerate bonds differently.

In [ ]:
def keyed(data: dict) -> dict:
    idx, val = data["idx"], data["val"]
    return {tuple(row): val[i] for i, row in enumerate(idx)}


def hermiticity_keyed(km: dict) -> tuple[float, int]:
    """max |A_nm(R) - conj(A_mn(-R))| / max|A|, matched by key; missing partners."""
    max_abs = max(np.abs(v).max() for v in km.values())
    worst, missing = 0.0, 0
    for (r1, r2, r3, n, m), v in km.items():
        partner = km.get((-r1, -r2, -r3, m, n))
        if partner is None:
            missing += 1
            continue
        worst = max(worst, np.abs(v[0::2] - partner[0::2]).max())
        worst = max(worst, np.abs(v[1::2] + partner[1::2]).max())
    return worst / max_abs, missing


def compare(candidate_path: Path, reference_path: Path, label: str) -> dict:
    cand, ref = read_matrix(candidate_path), read_matrix(reference_path)
    kc, kr = keyed(cand), keyed(ref)
    only_c, only_r = set(kc) - set(kr), set(kr) - set(kc)
    common = set(kc) & set(kr)
    diffs = np.array([np.abs(kc[k] - kr[k]).max() for k in common])
    ref_abs = np.array([np.abs(kr[k]).max() for k in common])
    rounded_ok = np.array([
        np.all(np.abs(kc[k] - fortran_round(kr[k], PRINT_STEP)) < 1e-12) for k in common
    ])
    big = ref_abs > 1e-5
    herm, missing_partners = hermiticity_keyed(kc)
    return dict(
        label=label, candidate_rows=len(kc), reference_rows=len(kr),
        only_in_candidate=len(only_c), only_in_reference=len(only_r),
        common_keys=len(common), max_abs_diff=float(diffs.max()),
        bit_exact_fraction=float(rounded_ok.mean()),
        max_relative_diff_above_1e5=float((diffs[big] / ref_abs[big]).max()) if big.any() else 0.0,
        hermiticity_candidate=float(herm), hermiticity_missing_partners=missing_partners,
    )


def report(result: dict) -> None:
    r = result
    print(f"\n=== {r['label']} ===")
    print(f"  candidate rows {r['candidate_rows']}  reference rows {r['reference_rows']}  "
          f"common keys {r['common_keys']}  "
          f"(only-candidate {r['only_in_candidate']}, only-reference {r['only_in_reference']})")
    print(f"  bit-exact after 6-decimal rounding  {r['bit_exact_fraction']:.6%}")
    print(f"  max |candidate - reference| over common keys  {r['max_abs_diff']:.3e}")
    print(f"  max rel. diff, |ref|>1e-5  {r['max_relative_diff_above_1e5']:.3e}")
    print(f"  Hermiticity, candidate (keyed)  {r['hermiticity_candidate']:.3e}  "
          f"({r['hermiticity_missing_partners']} rows with no -R partner)")

## Run each case

`results[label]` holds the full comparison dict (`bit_exact_fraction`, `max_abs_diff`, Hermiticity, ...) as a live object.

In [ ]:
if RUN_COMPUTATION:
    results = {}
    for label, candidate_path, reference_path in CASES:
        results[label] = compare(candidate_path, reference_path, label)
        report(results[label])
    passed = all(
        r["only_in_candidate"] == 0 and r["only_in_reference"] == 0 and r["bit_exact_fraction"] == 1.0
        for r in results.values()
    )
    print(f"\npassed = {passed}")

## Load the historical captures (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    for path in sorted(OUTPUT_DIR.glob("wsT_keyed_*.txt")):
        print(f"--- {path.name} ---")
        print(path.read_text())